# GLM-OCR vs PaddleOCR-VL-1.5 Benchmark

This notebook benchmarks **GLM-OCR** (0.9B) against **PaddleOCR-VL-1.5** (0.9B) across 8 diverse OCR datasets:

| # | Dataset | Task |
|---|---------|------|
| 1 | CAPTCHA | Distorted alphanumeric text recognition |
| 2 | LaTeX equations | Mathematical formula extraction |
| 3 | Receipts (total field) | Structured value extraction |
| 4 | Date stamps | Date extraction (YYYY-MM-DD) |
| 5 | Basketball jersey numbers | Short numeric OCR |
| 6 | Container serial numbers | Alphanumeric serial codes |
| 7 | Tire codes | Embossed alphanumeric codes |
| 8 | License plates | Vehicle plate recognition |

**Metrics**: Exact Match Accuracy, Character Error Rate (CER), Normalized Edit Distance (NED)  
**See** `ocr_metrics_research.md` for metric rationale.

---

## 1. Install Dependencies

In [ ]:
%%capture
!pip install transformers>=5.0.0 accelerate torch torchvision pillow
!pip install roboflow supervision
!pip install Levenshtein matplotlib seaborn pandas

## 2. Configuration & Secrets

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')

# Dataset slugs from Roboflow (workspace/project/version)
DATASETS = {
    "captcha":    {"workspace": "brad-dwyer",      "project": "captcha-hm1uu",             "version": 1, "split": "test"},
    "latex":      {"workspace": "roboflow-100",    "project": "latex-ocr",                 "version": 1, "split": "test"},
    "receipt":    {"workspace": "roboflow-100",    "project": "receipt-o4rdr",             "version": 1, "split": "test"},
    "datestamp":  {"workspace": "roboflow-100",    "project": "date-field-kfpf5",         "version": 1, "split": "test"},
    "jersey":     {"workspace": "roboflow-100",    "project": "jersey-number-recognition","version": 1, "split": "test"},
    "container":  {"workspace": "roboflow-100",    "project": "container-serial-number",  "version": 1, "split": "test"},
    "tire":       {"workspace": "roboflow-100",    "project": "tire-serial-number",       "version": 1, "split": "test"},
    "license":    {"workspace": "roboflow-100",    "project": "license-plate-recognition-rxg4e", "version": 1, "split": "test"},
}

# Prompts for each task (same for both models)
PROMPTS = {
    "captcha":   "What text is shown in this CAPTCHA image? Return only the exact characters, no explanation.",
    "latex":     "Extract the mathematical formula from this image as LaTeX. Return only the LaTeX code, no explanation.",
    "receipt":   "Extract the total amount from this receipt. Return only the numeric value (e.g. 45000), no currency symbol or explanation.",
    "datestamp": "What date is shown in this image? Return only the date in YYYY-MM-DD format, no explanation.",
    "jersey":    "What number is on this jersey? Return only the number, no explanation.",
    "container": "What is the serial number on this shipping container? Return only the serial number, no explanation.",
    "tire":      "What code is printed on this tire? Return only the code, no explanation.",
    "license":   "What is the license plate number in this image? Return only the plate text, no explanation.",
}

N_SAMPLES = 50  # number of test images per dataset (set to None for all)

## 3. Metric Utilities

In [ ]:
import Levenshtein
import re

def normalise(text: str) -> str:
    """Lowercase, strip whitespace."""
    return text.strip().lower()

def exact_match(pred: str, ref: str) -> int:
    return int(normalise(pred) == normalise(ref))

def cer(pred: str, ref: str) -> float:
    """Character Error Rate = edit_distance / len(reference)."""
    ref_n = normalise(ref)
    if len(ref_n) == 0:
        return 0.0 if len(normalise(pred)) == 0 else 1.0
    return Levenshtein.distance(normalise(pred), ref_n) / len(ref_n)

def ned(pred: str, ref: str) -> float:
    """Normalized Edit Distance = edit_distance / max(len(pred), len(ref))."""
    p, r = normalise(pred), normalise(ref)
    denom = max(len(p), len(r))
    if denom == 0:
        return 0.0
    return Levenshtein.distance(p, r) / denom

def evaluate(predictions: list[str], references: list[str]) -> dict:
    """Compute aggregate metrics for a list of predictions vs references."""
    assert len(predictions) == len(references)
    ems = [exact_match(p, r) for p, r in zip(predictions, references)]
    cers = [cer(p, r) for p, r in zip(predictions, references)]
    neds = [ned(p, r) for p, r in zip(predictions, references)]
    return {
        "exact_match": sum(ems) / len(ems),
        "cer": sum(cers) / len(cers),
        "ned": sum(neds) / len(neds),
        "n": len(ems),
    }

## 4. Dataset Loading

In [ ]:
import os
import json
from pathlib import Path
from PIL import Image
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def load_dataset(key: str, n: int = None) -> list[dict]:
    """Download dataset and return list of {image, label} dicts."""
    cfg = DATASETS[key]
    project = rf.workspace(cfg["workspace"]).project(cfg["project"])
    version = project.version(cfg["version"])
    dataset = version.download("folder", location=f"/tmp/rf_{key}")

    split_dir = Path(dataset.location) / cfg["split"]
    # Labels are stored alongside images as .txt files with the ground truth
    samples = []
    for img_path in sorted(split_dir.glob("*.jpg")) + sorted(split_dir.glob("*.png")):
        label_path = img_path.with_suffix(".txt")
        if not label_path.exists():
            continue
        label = label_path.read_text().strip()
        if key == "receipt":
            # Extract total field from JSON label
            label = extract_receipt_total(label)
        samples.append({"image_path": str(img_path), "label": label})

    if n is not None:
        samples = samples[:n]
    print(f"[{key}] loaded {len(samples)} samples")
    return samples

def extract_receipt_total(label_text: str) -> str:
    """Extract total field from receipt JSON label."""
    try:
        data = json.loads(label_text)
        # Try common field names
        for key in ["total", "total_price", "total_amount", "grand_total"]:
            if key in data:
                return str(data[key]).replace(",", "").replace(".", "").strip()
    except Exception:
        pass
    return label_text  # fallback to raw

## 5. Load Models

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoModelForImageTextToText

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print(f"Using device: {DEVICE}")

In [ ]:
# --- GLM-OCR ---
print("Loading GLM-OCR...")
GLM_MODEL_ID = "THUDM/glm-ocr"

glm_processor = AutoProcessor.from_pretrained(GLM_MODEL_ID, trust_remote_code=True)
glm_model = AutoModelForCausalLM.from_pretrained(
    GLM_MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE).eval()
print("GLM-OCR loaded.")

In [ ]:
# --- PaddleOCR-VL-1.5 ---
print("Loading PaddleOCR-VL-1.5...")
PADDLE_MODEL_ID = "PaddlePaddle/PaddleOCR-VL-1.5"

paddle_processor = AutoProcessor.from_pretrained(PADDLE_MODEL_ID, trust_remote_code=True)
paddle_model = AutoModelForImageTextToText.from_pretrained(
    PADDLE_MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE).eval()
print("PaddleOCR-VL-1.5 loaded.")

## 6. Inference Functions

In [ ]:
def run_glm_ocr(image: Image.Image, prompt: str) -> str:
    """Run GLM-OCR inference on a single image."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = glm_processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = glm_processor(text=text, images=[image], return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        output_ids = glm_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    # Decode only the newly generated tokens
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return glm_processor.decode(generated[0], skip_special_tokens=True).strip()


def run_paddle_ocr(image: Image.Image, prompt: str) -> str:
    """Run PaddleOCR-VL-1.5 inference on a single image."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    # PaddleOCR-VL-1.5 uses apply_chat_template too (Qwen2-VL compatible)
    text = paddle_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = paddle_processor(
        text=[text], images=[image], return_tensors="pt"
    ).to(DEVICE)
    with torch.inference_mode():
        output_ids = paddle_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    return paddle_processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

## 7. Run Benchmark

This cell runs both models on all 8 datasets and stores per-sample predictions.

In [ ]:
from tqdm.auto import tqdm

results = {}  # key -> {"glm": {...}, "paddle": {...}}

for dataset_key in DATASETS:
    print(f"\n{'='*50}")
    print(f"Dataset: {dataset_key}")
    print(f"{'='*50}")

    samples = load_dataset(dataset_key, n=N_SAMPLES)
    prompt = PROMPTS[dataset_key]

    glm_preds, paddle_preds, labels = [], [], []

    for sample in tqdm(samples, desc=dataset_key):
        image = Image.open(sample["image_path"]).convert("RGB")
        label = sample["label"]

        glm_pred = run_glm_ocr(image, prompt)
        paddle_pred = run_paddle_ocr(image, prompt)

        glm_preds.append(glm_pred)
        paddle_preds.append(paddle_pred)
        labels.append(label)

    glm_metrics = evaluate(glm_preds, labels)
    paddle_metrics = evaluate(paddle_preds, labels)

    results[dataset_key] = {
        "glm": {**glm_metrics, "predictions": glm_preds},
        "paddle": {**paddle_metrics, "predictions": paddle_preds},
        "labels": labels,
    }

    print(f"  GLM-OCR     | EM={glm_metrics['exact_match']:.3f}  CER={glm_metrics['cer']:.3f}  NED={glm_metrics['ned']:.3f}")
    print(f"  PaddleOCR   | EM={paddle_metrics['exact_match']:.3f}  CER={paddle_metrics['cer']:.3f}  NED={paddle_metrics['ned']:.3f}")

print("\nBenchmark complete!")

## 8. Results Summary Table

In [ ]:
import pandas as pd

rows = []
for ds, res in results.items():
    for model_name, model_key in [("GLM-OCR", "glm"), ("PaddleOCR-VL-1.5", "paddle")]:
        m = res[model_key]
        rows.append({
            "Dataset": ds,
            "Model": model_name,
            "Exact Match": round(m["exact_match"], 4),
            "CER": round(m["cer"], 4),
            "NED": round(m["ned"], 4),
            "N": m["n"],
        })

df = pd.DataFrame(rows)
df_pivot = df.pivot(index="Dataset", columns="Model", values=["Exact Match", "CER", "NED"])
print(df_pivot.to_string())

## 9. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

DATASET_LABELS = {
    "captcha":   "CAPTCHA",
    "latex":     "LaTeX",
    "receipt":   "Receipt\n(total)",
    "datestamp": "Date\nStamp",
    "jersey":    "Jersey\nNumber",
    "container": "Container\nSerial",
    "tire":      "Tire Code",
    "license":   "License\nPlate",
}

datasets = list(results.keys())
x = np.arange(len(datasets))
width = 0.35
xtick_labels = [DATASET_LABELS[d] for d in datasets]

COLORS = {"GLM-OCR": "#4C72B0", "PaddleOCR-VL-1.5": "#DD8452"}

In [ ]:
# --- Plot 1: Exact Match Accuracy (grouped bar) ---
fig, ax = plt.subplots(figsize=(12, 5))

glm_em = [results[d]["glm"]["exact_match"] for d in datasets]
paddle_em = [results[d]["paddle"]["exact_match"] for d in datasets]

bars1 = ax.bar(x - width/2, glm_em, width, label="GLM-OCR", color=COLORS["GLM-OCR"], alpha=0.85)
bars2 = ax.bar(x + width/2, paddle_em, width, label="PaddleOCR-VL-1.5", color=COLORS["PaddleOCR-VL-1.5"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, fontsize=10)
ax.set_ylabel("Exact Match Accuracy", fontsize=12)
ax.set_title("Exact Match Accuracy by Dataset", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt="%.0f%%", label_type="edge", fontsize=8, padding=2,
             labels=[f"{v*100:.0f}%" for v in glm_em])
ax.bar_label(bars2, fmt="%.0f%%", label_type="edge", fontsize=8, padding=2,
             labels=[f"{v*100:.0f}%" for v in paddle_em])
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("plot_exact_match.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 2: Character Error Rate (lower is better) ---
fig, ax = plt.subplots(figsize=(12, 5))

glm_cer = [results[d]["glm"]["cer"] for d in datasets]
paddle_cer = [results[d]["paddle"]["cer"] for d in datasets]

bars1 = ax.bar(x - width/2, glm_cer, width, label="GLM-OCR", color=COLORS["GLM-OCR"], alpha=0.85)
bars2 = ax.bar(x + width/2, paddle_cer, width, label="PaddleOCR-VL-1.5", color=COLORS["PaddleOCR-VL-1.5"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, fontsize=10)
ax.set_ylabel("Character Error Rate (lower = better)", fontsize=12)
ax.set_title("Character Error Rate by Dataset", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("plot_cer.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 3: Radar / Spider chart — overall profile ---
from matplotlib.patches import FancyArrowPatch

# Use (1 - NED) as a similarity score so higher = better for all axes
glm_scores = [results[d]["glm"]["exact_match"] for d in datasets]
paddle_scores = [results[d]["paddle"]["exact_match"] for d in datasets]
short_labels = [DATASET_LABELS[d].replace("\n", " ") for d in datasets]

angles = np.linspace(0, 2 * np.pi, len(datasets), endpoint=False).tolist()
glm_scores += glm_scores[:1]
paddle_scores += paddle_scores[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, glm_scores, "o-", linewidth=2, color=COLORS["GLM-OCR"], label="GLM-OCR")
ax.fill(angles, glm_scores, alpha=0.15, color=COLORS["GLM-OCR"])
ax.plot(angles, paddle_scores, "s-", linewidth=2, color=COLORS["PaddleOCR-VL-1.5"], label="PaddleOCR-VL-1.5")
ax.fill(angles, paddle_scores, alpha=0.15, color=COLORS["PaddleOCR-VL-1.5"])

ax.set_thetagrids(np.degrees(angles[:-1]), short_labels, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7)
ax.set_title("Exact Match Accuracy — Radar Overview", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plot_radar.png", dpi=150)
plt.show()

In [ ]:
# --- Plot 4: Summary — average across all datasets ---
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

metrics_info = [
    ("exact_match", "Avg. Exact Match", True),
    ("cer",         "Avg. CER",         False),
    ("ned",         "Avg. NED",         False),
]

model_keys = [("GLM-OCR", "glm"), ("PaddleOCR-VL-1.5", "paddle")]

for ax, (metric_key, metric_label, higher_better) in zip(axes, metrics_info):
    vals = [
        np.mean([results[d][mk][metric_key] for d in datasets])
        for _, mk in model_keys
    ]
    model_names = [mn for mn, _ in model_keys]
    colors = [COLORS[mn] for mn in model_names]
    bars = ax.bar(model_names, vals, color=colors, alpha=0.85, width=0.4)
    ax.set_title(metric_label, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 + 0.01)
    note = "↑ higher better" if higher_better else "↓ lower better"
    ax.set_xlabel(note, fontsize=9, color="grey")
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=10)

fig.suptitle("Overall Average Performance (all 8 datasets)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Per-Dataset Qualitative Sample Preview

Show a few sample predictions side by side for manual inspection.

In [ ]:
N_PREVIEW = 3  # samples per dataset to show

for dataset_key in datasets:
    print(f"\n{'━'*60}")
    print(f"  {DATASET_LABELS[dataset_key].replace(chr(10), ' ')}")
    print(f"{'━'*60}")
    res = results[dataset_key]
    for i in range(min(N_PREVIEW, len(res["labels"]))):
        gt = res["labels"][i]
        g = res["glm"]["predictions"][i]
        p = res["paddle"]["predictions"][i]
        g_em = "✓" if normalise(g) == normalise(gt) else "✗"
        p_em = "✓" if normalise(p) == normalise(gt) else "✗"
        print(f"  [{i+1}] GT:     {gt}")
        print(f"       GLM:    {g_em} {g}")
        print(f"       Paddle: {p_em} {p}")
        print()